# Fresh vs Rotten Apple Classification — Transfer Learning with MobileNetV2

**GET 324 — Laboratory Exercise 10 (Mini-Project)**
**Assigned task (CV10):** Fresh Apple vs Rotten Apple (binary image classification)
**Approach:** Transfer learning using MobileNetV2 pretrained on ImageNet

This notebook covers the model-development portion of the mini-project:

1. Environment setup & dataset discovery
2. Data loading, exploration and preprocessing
3. Data augmentation
4. Model building (MobileNetV2 base + custom classification head)
5. Training — Phase 1: frozen feature extraction
6. Training — Phase 2: fine-tuning the top layers
7. Training curves
8. Evaluation (accuracy, precision, recall, confusion matrix, ROC-AUC)
9. Saving the trained model as `model.keras`
10. Quick inference demo on sample images

> **Before running:** in the Kaggle notebook editor, open **Settings → Accelerator** and select a GPU
> (T4 x2 or P100) — training MobileNetV2 on CPU will be much slower. Also confirm the dataset is
> attached under **Add Input** (it should already be, per your setup).

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


## 1. Locate the Dataset

The dataset is attached to this Kaggle notebook under `/kaggle/input/`. The cell below automatically
walks the input directory and locates the `train`, `validation` (or `val`) and `test` folders, so you
don't need to hardcode a path that could break if the dataset is re-added or renamed.

In [ ]:
def find_split_dirs(root="/kaggle/input"):
    """Walk the Kaggle input directory and locate train/validation/test folders."""
    train_dir = val_dir = test_dir = None
    for dirpath, dirnames, filenames in os.walk(root):
        base = os.path.basename(dirpath).lower()
        if base in ("train", "training") and train_dir is None:
            train_dir = dirpath
        elif base in ("validation", "val", "valid") and val_dir is None:
            val_dir = dirpath
        elif base in ("test", "testing") and test_dir is None:
            test_dir = dirpath
    return train_dir, val_dir, test_dir

train_dir, val_dir, test_dir = find_split_dirs()

assert train_dir is not None, (
    "Could not find a 'train' folder under /kaggle/input. "
    "Check that the dataset is attached (Settings > Input)."
)
assert test_dir is not None, "Could not find a 'test' folder under /kaggle/input."

print("Train directory:     ", train_dir)
print("Validation directory:", val_dir)
print("Test directory:      ", test_dir)
print("\nClasses found in train folder:", sorted(os.listdir(train_dir)))


In [ ]:
def count_images(directory):
    counts = {}
    if directory is None:
        return counts
    for class_name in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            counts[class_name] = len([
                f for f in os.listdir(class_path)
                if f.lower().endswith((".png", ".jpg", ".jpeg"))
            ])
    return counts

print("Train counts:     ", count_images(train_dir))
print("Validation counts:", count_images(val_dir))
print("Test counts:      ", count_images(test_dir))


## 2. Load Data with `image_dataset_from_directory`

We build `tf.data.Dataset` pipelines directly from the folder structure. Keras automatically infers
the two class labels from the subfolder names (alphabetical order → index 0, index 1). Since this is
binary classification, `label_mode="binary"` gives labels of shape `(batch, 1)` with values 0.0/1.0,
which pairs with a sigmoid output layer and `binary_crossentropy` loss.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

if val_dir is not None:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, labels="inferred", label_mode="binary",
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED,
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir, labels="inferred", label_mode="binary",
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
    )
else:
    # No separate validation folder found: carve 20% out of the training set instead
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, labels="inferred", label_mode="binary", image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, shuffle=True, seed=SEED,
        validation_split=0.2, subset="training",
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, labels="inferred", label_mode="binary", image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, shuffle=False, seed=SEED,
        validation_split=0.2, subset="validation",
    )

class_names = train_ds.class_names
print("Class names (index 0 / index 1):", class_names)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, labels="inferred", label_mode="binary",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
)


In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_idx = int(labels[i].numpy()[0])
        plt.title(class_names[label_idx])
        plt.axis("off")
plt.suptitle("Sample Training Images")
plt.tight_layout()
plt.show()


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


## 3. Data Augmentation

Light augmentation (flips, rotation, zoom, contrast) is applied **only during training**, as Keras
layers built into the model graph itself. This reduces overfitting without a separate preprocessing
pipeline, and it is automatically skipped at evaluation/inference time (`training=False`).

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name="data_augmentation")


## 4. Build the Model — MobileNetV2 Transfer Learning

MobileNetV2, pretrained on ImageNet, is used as a frozen feature extractor, with a small custom
classification head attached on top for the fresh/rotten decision. `preprocess_input` rescales pixels
to the `[-1, 1]` range MobileNetV2 was trained on.

In [ ]:
IMG_SHAPE = IMG_SIZE + (3,)

base_model = MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # freeze for the initial feature-extraction phase

inputs = keras.Input(shape=IMG_SHAPE)
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs, name="mobilenetv2_apple_classifier")
model.summary()


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ],
)


## 5. Phase 1 — Train the Classification Head (base frozen)

In [ ]:
callbacks_phase1 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("best_model_phase1.keras", monitor="val_accuracy", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
]

EPOCHS_PHASE1 = 15

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_phase1,
)


## 6. Phase 2 — Fine-Tune the Top Layers of MobileNetV2

We now unfreeze the top portion of the base model and continue training with a much smaller learning
rate. This lets the network adapt its higher-level features — previously tuned for the 1000 ImageNet
classes — specifically to fresh vs. rotten apple textures, without destroying the low-level filters
(edges, colors, textures) learned from ImageNet.

In [ ]:
base_model.trainable = True

# Freeze all layers except the last N — keeps low-level ImageNet features intact
FINE_TUNE_AT = len(base_model.layers) - 40
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ],
)

EPOCHS_PHASE2 = 10
total_epochs = EPOCHS_PHASE1 + EPOCHS_PHASE2

callbacks_phase2 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("best_model_finetuned.keras", monitor="val_accuracy", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-8),
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history1.epoch[-1] + 1,
    callbacks=callbacks_phase2,
)


## 7. Training Curves

In [ ]:
def combine(h1, h2, key):
    return h1.history[key] + h2.history[key]

acc = combine(history1, history2, "accuracy")
val_acc = combine(history1, history2, "val_accuracy")
loss = combine(history1, history2, "loss")
val_loss = combine(history1, history2, "val_loss")

epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Train Accuracy")
plt.plot(epochs_range, val_acc, label="Validation Accuracy")
plt.axvline(x=EPOCHS_PHASE1, color="gray", linestyle="--", label="Fine-tuning starts")
plt.legend(loc="lower right")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Train Loss")
plt.plot(epochs_range, val_loss, label="Validation Loss")
plt.axvline(x=EPOCHS_PHASE1, color="gray", linestyle="--", label="Fine-tuning starts")
plt.legend(loc="upper right")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.tight_layout()
plt.show()


## 8. Evaluate on the Held-Out Test Set

In [ ]:
test_loss, test_acc, test_prec, test_rec, test_auc = model.evaluate(test_ds)
print(f"Test Loss:      {test_loss:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall:    {test_rec:.4f}")
print(f"Test AUC:       {test_auc:.4f}")


In [ ]:
y_true = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0).ravel()
y_pred_prob = model.predict(test_ds).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                  color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()


## 9. Save the Trained Model

In [ ]:
model.save("model.keras")
print("Model saved to:", os.path.abspath("model.keras"))
print("File size (MB):", round(os.path.getsize("model.keras") / (1024 * 1024), 2))


## 10. Quick Inference Demo

Reloads the saved model fresh (as your deployed app would) and runs predictions on a handful of test
images, so you can sanity-check the model before wiring it into the application-development and
cloud-deployment stages of the assignment.

In [ ]:
loaded_model = keras.models.load_model("model.keras")

def predict_image(model, img_path, class_names, img_size=IMG_SIZE):
    img = keras.utils.load_img(img_path, target_size=img_size)
    img_array = keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    prob = model.predict(img_array, verbose=0)[0][0]
    pred_idx = int(prob >= 0.5)
    return class_names[pred_idx], prob

# Pull a few sample images straight from the test folders
sample_paths = []
for class_name in class_names:
    class_dir = os.path.join(test_dir, class_name)
    files = [f for f in os.listdir(class_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))][:3]
    sample_paths += [os.path.join(class_dir, f) for f in files]

plt.figure(figsize=(12, 8))
for i, path in enumerate(sample_paths[:9]):
    label, prob = predict_image(loaded_model, path, class_names)
    img = keras.utils.load_img(path, target_size=IMG_SIZE)
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(f"{label} ({prob:.2f})")
    plt.axis("off")
plt.tight_layout()
plt.show()


## Summary

- **Model:** MobileNetV2 (ImageNet weights) + custom dense head, trained in two phases — frozen
  feature extraction, then fine-tuning the top ~40 layers.
- **Output:** `model.keras`, saved to `/kaggle/working/` — ready to load with
  `keras.models.load_model("model.keras")` for the application-development and cloud-deployment
  stages of this assignment.
- **Next steps for the group:** wrap `predict_image()` (or the loaded model) in a small web app
  (e.g. Streamlit/Flask/FastAPI), then deploy it to a cloud platform per the assignment brief.

**If accuracy is lower than you'd like, try:** more/better-augmented training data, unfreezing more
base-model layers during fine-tuning, a lower fine-tuning learning rate, more epochs, or checking the
confusion matrix above for which class is being confused most.